In [1]:
import os 


os.environ.setdefault("LLM_MODEL", "openai:gpt-4.1")

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set")


In [2]:
from langchain_core.documents import Document 
from langchain.chat_models import init_chat_model

In [5]:
# Sample non-parametric memory facts
FACTS = [
    "Zorvia Coffee annual subscriptions can be refunded within 14 days of purchase.",
    "Monthly subscriptions can be cancelled at any time, but the current month is not refunded.",
    "Orders of 5 bags or more ship free.",
    "Zorvia roasts every Tuesday and Friday in Lisbon.",
    "Customer support is open 9:00 to 17:00 UTC, Monday to Friday.",
    "The Ember blend is a dark roast with notes of cocoa and orange peel.",
    "The Drift blend is a light roast from Ethiopia with notes of jasmine and lemon.",
    "Zorvia gift cards never expire.",
    "Beans are freshest for 30 days after the roast date if kept in a sealed bag.",
    "Wholesale accounts require a minimum order of 20 kg per month.",
    "In the referral program, both the referrer and the friend get one free bag.",
    "Orders placed before 12:00 UTC ship the same day.",
]

model = init_chat_model(os.environ.get("LLM_MODEL"))

In [4]:
def to_documents(facts: list[str]) -> list[Document]:
    return [Document(page_content=text, metadata={"id": i}) for i, text in enumerate(facts)]

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", # 384D embeddings
    encode_kwargs={"normalize_embeddings": True} # l2 norm
)

/Users/sanket/Developer/Codebases/AiEngineeringConcepts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 42525.18it/s]


In [12]:
from langchain_core.vectorstores import InMemoryVectorStore

def build_store(facts: list[Document]) -> InMemoryVectorStore:
    store = InMemoryVectorStore(embedding=embeddings_model) # this creates the vector store using sentence-transformers/all-MiniLM-L6-v2 embeddings
    store.add_documents(facts) # this adds the documents to the vector store i.e. document -> embedding -> stored in the vector store
    return store 


In [13]:
v_store = build_store(to_documents(FACTS))

In [14]:
sample_query = "Refund policy?"

sample_vec = embeddings_model.embed_query(sample_query)

print(len(sample_vec), sample_vec[:6])

384 [-0.08512810617685318, 0.055443692952394485, 0.060503289103507996, -0.0010946643305942416, 0.04262227192521095, 0.04242919012904167]


In [16]:
QUESTION = "How long do I have to ask for a refund on my annual plan ?"

# the top k closest chunks with their similarity scores

for chunk, score in v_store.similarity_search_with_score(QUESTION, k=3):
    print(f"Score: {score:.2f}")
    print(chunk.page_content)
    print(chunk.metadata)
    print("\n")



Score: 0.44
Zorvia Coffee annual subscriptions can be refunded within 14 days of purchase.
{'id': 0}


Score: 0.40
Monthly subscriptions can be cancelled at any time, but the current month is not refunded.
{'id': 1}


Score: 0.23
Wholesale accounts require a minimum order of 20 kg per month.
{'id': 9}




In [17]:
def answer(query, store, k=3):
    relevant_facts = store.similarity_search(query, k=k)
    context = "\n".join(f"- {d.page_content}" for d in relevant_facts)
    prompt = (
        "Answer the following question based on the notes provided.\n"
        "If the notes don't contain the answer, say 'No information found'.\n"
        f"Notes: \n{context}\n\nQuestion: {query}:"
    )
    return model.invoke(prompt).text

In [18]:
print("Direct answer without rag: ", model.invoke(QUESTION).text)

Direct answer without rag:  The refund policy for your **annual plan** depends on the company or service you purchased from, as different companies have different terms.

**Common Scenarios:**
- **Many SaaS companies (e.g., Adobe, Microsoft, some cloud services):** Often offer a **14-day window** for a full refund after purchase or renewal.
- **Others (like streaming services, domains, or web hosting):** May offer a **30-day money-back guarantee**.
- Some companies may **not offer refunds** once the annual plan starts, or after a certain period.

**What to do:**
1. **Check the Terms of Service or Refund Policy:** This is usually found in your account portal or on the company’s website.
2. **Contact Customer Support:** If you’re unsure, reach out directly and ask about your eligibility and the refund deadline.

**If you tell me the specific service or company, I can provide more precise information.**


In [19]:
print("Answer with rag: ", answer(QUESTION, v_store))

Answer with rag:  You have 14 days from purchase to ask for a refund on your annual plan.


In [23]:
FACTS[0] = "Zorvia Coffee annual subscriptions can be refunded within 30 days of purchase."

v_store = build_store(to_documents(FACTS))

In [24]:
print("Answer with rag: ", answer(QUESTION, v_store))

Answer with rag:  You have 30 days from purchase to ask for a refund on your annual plan.


In [25]:
print("Answer unaware question: ", answer("What is the weather in Lisbon?", v_store))

Answer unaware question:  No information found.
